# Shortest proof: fine-tuning lifts next-action accuracy

Task: given a shopping page (OPeRA HTML), predict the human's next action  
`click(name)` / `type_and_submit(name, text)` / `terminate()`.

Loop: **base Qwen-0.5B → LoRA on a train slice → same held-out sessions**.  
Metric: session-macro exact-match (paper Table 2). Also report parse rate so a format-only lift is visible.

**Colab:** Runtime → T4 GPU → Run all. About 20–30 minutes.  
Upload this repo (needs `opera_repro/`) or open the notebook from the project root.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q "transformers>=4.45" "peft>=0.13" "accelerate>=1.0" "datasets>=3.0" matplotlib

ROOT = Path.cwd()
for cand in [ROOT, ROOT.parent, Path("/content"), *ROOT.glob("**/")]:
    if (cand / "opera_repro" / "__init__.py").exists():
        ROOT = cand
        break
sys.path.insert(0, str(ROOT))
print("repo", ROOT)
assert (ROOT / "opera_repro").exists(), "Upload the project so opera_repro/ is on the path."

In [ ]:
import torch

assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU, then rerun."
props = torch.cuda.get_device_properties(0)
print(torch.cuda.get_device_name(0), round(props.total_memory / 1e9, 1), "GB")

## Data

OPeRA-filtered, official session split, then a Colab-sized cap. Splits happen **before** examples so no session leaks into test.

In [ ]:
from collections import Counter

from opera_repro.converter import ConverterConfig, convert_rows, split_stats
from opera_repro.data import load_opera_filtered
from opera_repro.evaluate import evaluate_predictions, format_report

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_TRAIN_SESSIONS = 80
MAX_TEST_SESSIONS = 20
MAX_TRAIN_EX = 320
MAX_TEST_EX = 80
MAX_SEQ = 1024
EPOCHS = 2
MAX_NEW = 64

cfg = ConverterConfig(
    split_mode="official",
    max_history_steps=2,
    max_current_html_chars=1200,
    max_history_html_chars=400,
)
rows = load_opera_filtered(cache_dir=str(ROOT / "data" / "cache"))
print("raw actions", len(rows))
examples = convert_rows(rows, cfg)

def cap_by_session(split_rows, n_sessions, n_ex):
    seen, keep = [], []
    for row in split_rows:
        sid = row["session_id"]
        if sid not in seen:
            if len(seen) >= n_sessions:
                continue
            seen.append(sid)
        if sid in seen:
            keep.append(row)
        if len(keep) >= n_ex:
            break
    return keep

train = cap_by_session(examples["train"], MAX_TRAIN_SESSIONS, MAX_TRAIN_EX)
test = cap_by_session(examples["test"], MAX_TEST_SESSIONS, MAX_TEST_EX)
print(split_stats({"train": train, "test": test}))
print("train types", Counter(r["gold_action"]["type"] for r in train))
print("test types ", Counter(r["gold_action"]["type"] for r in test))

In [ ]:
import matplotlib.pyplot as plt

types = ["click", "type_and_submit", "terminate"]
tr = Counter(r["gold_action"]["type"] for r in train)
te = Counter(r["gold_action"]["type"] for r in test)
x = range(len(types))
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar([i - 0.18 for i in x], [tr[t] for t in types], width=0.36, label="train", color="#3b6ea5")
ax.bar([i + 0.18 for i in x], [te[t] for t in types], width=0.36, label="test", color="#c47b2b")
ax.set_xticks(list(x), types)
ax.set_ylabel("examples")
ax.set_title("Gold next-action types (session-capped OPeRA slice)")
ax.legend()
fig.tight_layout()
plt.show()

## Baselines then LoRA

Majority type is a naive floor. Base Qwen is the prompt-only model. LoRA is the only trained weights.

In [ ]:
import json
from collections import defaultdict

from opera_repro.actions import parse_action

majority_type = Counter(r["gold_action"]["type"] for r in train).most_common(1)[0][0]
majority_preds = [json.dumps({"type": majority_type}) for _ in test]
majority = evaluate_predictions(test, majority_preds)
print(format_report(majority, "majority-type baseline"))

scores = {
    "majority": majority,
}

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("params", round(model.num_parameters() / 1e6, 1), "M")


@torch.no_grad()
def predict(records, mdl):
    mdl.eval()
    out = []
    for i, row in enumerate(records, start=1):
        messages = [m for m in row["messages"] if m["role"] != "assistant"]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ).to(mdl.device)
        gen = mdl.generate(
            **inputs,
            max_new_tokens=MAX_NEW,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        text = tokenizer.decode(gen[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
        out.append(text)
        if i % 20 == 0 or i == len(records):
            print(f"  {i}/{len(records)}")
    return out

In [ ]:
print("base eval…")
base_preds = predict(test, model)
base = evaluate_predictions(test, base_preds)
scores["base"] = base
print(format_report(base, "Qwen2.5-0.5B base"))
print("parse rate", 1 - base.n_illegal / base.n_examples)

In [ ]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments, default_data_collator

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
model.train()

def tokenize_row(row):
    text = tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)
    tok = tokenizer(text, truncation=True, max_length=MAX_SEQ, padding="max_length")
    tok["labels"] = [(tid if tid != tokenizer.pad_token_id else -100) for tid in tok["input_ids"]]
    return tok

ds = Dataset.from_list([{"messages": r["messages"]} for r in train]).map(tokenize_row, remove_columns=["messages"])
args = TrainingArguments(
    output_dir=str(ROOT / "outputs" / "colab-qwen05-opera-lora"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    fp16=True,
    report_to="none",
    remove_unused_columns=False,
)
trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=default_data_collator)
trainer.train()
adapter_dir = ROOT / "outputs" / "colab-qwen05-opera-lora"
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(adapter_dir)
print("saved", adapter_dir)

In [ ]:
print("finetuned eval…")
ft_preds = predict(test, model)
ft = evaluate_predictions(test, ft_preds)
scores["finetuned"] = ft
print(format_report(ft, "Qwen2.5-0.5B + LoRA"))
print("parse rate", 1 - ft.n_illegal / ft.n_examples)
print(
    "lift session-macro",
    f"{base.session_macro_accuracy:.2%} → {ft.session_macro_accuracy:.2%}",
)

In [ ]:
labels = ["majority", "base", "finetuned"]
macro = [scores[k].session_macro_accuracy * 100 for k in labels]
atype = [scores[k].action_type_accuracy * 100 for k in labels]
parse = [(1 - scores[k].n_illegal / scores[k].n_examples) * 100 for k in labels]

fig, ax = plt.subplots(figsize=(7, 3.6))
x = range(len(labels))
w = 0.25
ax.bar([i - w for i in x], macro, width=w, label="session-macro exact %", color="#1f4e79")
ax.bar(list(x), atype, width=w, label="action-type %", color="#5b8c5a")
ax.bar([i + w for i in x], parse, width=w, label="valid JSON parse %", color="#b85c38")
ax.set_xticks(list(x), labels)
ax.set_ylabel("percent")
ax.set_ylim(0, 100)
ax.set_title("Held-out OPeRA next-action (same test sessions)")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

print(
    json.dumps(
        {
            k: {
                "session_macro": round(v.session_macro_accuracy, 4),
                "action_type": round(v.action_type_accuracy, 4),
                "parse_rate": round(1 - v.n_illegal / v.n_examples, 4),
                "n": v.n_examples,
            }
            for k, v in scores.items()
        },
        indent=2,
    )
)

In [ ]:
print(f"{'gold':<42} {'base':<42} {'ft'}")
for row, b, f in list(zip(test, base_preds, ft_preds))[:8]:
    gold = json.dumps(row["gold_action"], ensure_ascii=False)
    bp = parse_action(b)
    fp = parse_action(f)
    print(
        f"{(gold[:40]):<42} {(str(bp.to_dict() if bp else b[:30])):<42} {fp.to_dict() if fp else f[:30]}"
    )

### Q&A

Does fine-tuning meaningfully improve an actual task? After Run all, the printed lift (`base → finetuned` session-macro exact) is the answer. Ping this chat with that table if you want the numbers written into the demo.

### What this notebook is allowed to claim

- Same task the product uses: next human web action on real OPeRA pages.
- Same test sessions before and after. No leakage by construction.
- 0.5B + 320 examples is a *directional* Colab proof, not the paper's 7B / full split (4.10% → 32.04%).

### Next

- If parse rate jumps but exact-match stays low: the model learned JSON, not names — raise HTML budget / train size, or swap in Pioneer 7B.
- If exact-match jumps: drop the adapter into `simulator/agent.py` `_decide`.